# Phase Identification and Window Segmentation — PhaseNet Method

This notebook implements phase detection using **PhaseNet**, a deep-learning phase picker trained on the **INSTANCE dataset** (Michelini et al., 2021). PhaseNet uses convolutional neural networks to detect P and S arrivals directly from waveforms, without requiring pre-computed features or search windows.

**Comparison with AR-AIC (notebook 03a):**
- **AR-AIC:** Traditional method, requires search windows, processes 3 components together
- **PhaseNet:** Deep learning, no search windows needed, processes each component independently

**Workflow:**
1. **Load PhaseNet model** — Pre-trained INSTANCE model via SeisBench
2. **Resample signals** — 200 Hz → 100 Hz (PhaseNet training rate)
3. **Phase detection** — CNN-based P/S picking on each component
4. **Merge with metadata** — Join picks with station information
5. **Coda onset detection** — Same 4 methods as AR-AIC (Rautian, Arias, Envelope, Median)
6. **Window segmentation** — Create 4 windows: pre-event, P-wave, S-wave, coda
7. **Quality control** — Validate detections (monotonicity, SNR, peak timing)

**Key differences from 03a:**
- No CRUST1.0 queries needed (PhaseNet doesn't use velocity model)
- No adaptive search windows (CNN detects onsets directly)
- Component-level picks (unlike AR-AIC which uses all 3 components together)
- Resampling required (PhaseNet trained on 100 Hz data)

**Dataset:** Same preprocessed signals as notebook 03a (baseline-corrected, unnormalized).

**Key references:**
- Zhu & Beroza (2019) — PhaseNet original paper
- Woollam et al. (2022) — SeisBench framework
- Michelini et al. (2021) — INSTANCE dataset

**Outputs:**
- `df_full_{signal_type}_phasenet.parquet` — Complete metadata with PhaseNet picks
- `windowed_{signal_type}_{coda_method}_phasenet.pkl` — Segmented signals (4 files)
- `phasenet_picks_{signal_type}.csv` — Raw PhaseNet output
- Validation plots in `figures/03b_phase_identification_phasenet/{signal_type}/`

## 1. Imports and visualization settings

In [ ]:
import seisbench
import seisbench.models as sbm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import sys
import pickle
from pathlib import Path
from src  import (
    apply_phasenet_to_signals,
    set_plot_style,
    add_hypocentral_distance,
    expand_to_component_level,
    add_coda_onsets_to_dataframe,
    segment_all_signals,
    quality_control_all_stations,
    print_quality_control_summary,
    add_time_columns,
    convert_signals_to_dict,
    validate_signals_dict,
    merge_phasenet_picks_with_metadata,
    add_coda_end_to_dataframe,
    get_station_from_filename,
    plot_onset_detection_results,
    plot_multiple_stations,
    add_crustal_velocities,
    add_theoretical_arrivals
)
colors, colors1 = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

## 2. Configuration

Set `DATA_TYPE` to process acceleration, velocity, or displacement signals. The configuration automatically determines:
- Signal column name
- Physical units
- Peak ground motion columns (PGA/PGV/PGD)

PhaseNet can be applied to all three signal types, though it was originally trained on velocity waveforms.

In [ ]:
# CONFIGURATION
EVENT_ID = 'INT-41004391'
# EVENT_ID = 'IT-2009-0009'
DATA_TYPE = 'displacement'  # Options: 'acceleration', 'velocity', 'displacement'

# Determine signal column name and units based on DATA_TYPE
if DATA_TYPE == 'acceleration':
    SIGNAL_COLUMN = 'acceleration'
    SIGNAL_UNIT = 'cm/s²'
    PEAK_COLUMN = 'PGA_CM/S^2'
    TIME_PEAK_COLUMN = 'TIME_PGA_S'
elif DATA_TYPE == 'velocity':
    SIGNAL_COLUMN = 'velocity'
    SIGNAL_UNIT = 'cm/s'
    PEAK_COLUMN = 'PGV_CM/S'
    TIME_PEAK_COLUMN = 'TIME_PGV_S'
elif DATA_TYPE == 'displacement':
    SIGNAL_COLUMN = 'displacement'
    SIGNAL_UNIT = 'cm'
    PEAK_COLUMN = 'PGD_CM'
    TIME_PEAK_COLUMN = 'TIME_PGD_S'
else:
    raise ValueError(f"Unknown DATA_TYPE: {DATA_TYPE}")

logger.info(f"Working with {DATA_TYPE} data")
logger.info(f"Signal column: {SIGNAL_COLUMN}")
logger.info(f"Peak column: {PEAK_COLUMN}")

## 3. Data Loading

Load preprocessed signals and metadata:
- **Signals:** Baseline-corrected but **not normalized** (preserves waveform morphology for PhaseNet)
- **Metadata:** Station coordinates, epicentral distances, event parameters

**Note:** Unlike traditional pickers, PhaseNet can work with normalized signals, but unnormalized data preserves amplitude information that may improve performance.

In [ ]:
# Get project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Define all paths from project root with DATA_TYPE subdirectories
METADATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / EVENT_ID / '01a_metadata' / DATA_TYPE
SIGNALS_PROCESSED_IMPORT = PROJECT_ROOT / 'data' / 'processed' / EVENT_ID / '01b_signals' / DATA_TYPE
SIGNALS_PROCESSED_EXPORT = PROJECT_ROOT / 'data' / 'processed' / EVENT_ID / '03b_phase_identification_phasenet' / DATA_TYPE
FIGURES_DIR = PROJECT_ROOT / 'figures' / EVENT_ID / '03b_phase_identification_phasenet' / DATA_TYPE
LATEX_TABLES_DIR = PROJECT_ROOT / 'data' / 'processed' / EVENT_ID / 'latex_tables' / DATA_TYPE

# Create output directories
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_PROCESSED.mkdir(parents=True, exist_ok=True)
SIGNALS_PROCESSED_IMPORT.mkdir(parents=True, exist_ok=True)
SIGNALS_PROCESSED_EXPORT.mkdir(parents=True, exist_ok=True)
LATEX_TABLES_DIR.mkdir(parents=True, exist_ok=True)

check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")
check(SIGNALS_PROCESSED_EXPORT.exists(), f"Exported signals directory ready: {SIGNALS_PROCESSED_EXPORT}")
check(LATEX_TABLES_DIR.exists(), f"LaTeX tables directory ready: {LATEX_TABLES_DIR}")
check(METADATA_PROCESSED.exists(), f"Processed metadata directory ready: {METADATA_PROCESSED}")
check(SIGNALS_PROCESSED_IMPORT.exists(), f"Processed signals directory ready: {SIGNALS_PROCESSED_IMPORT}")

# Load metadata
logger.info(f"Loading {DATA_TYPE} metadata...")
df_meta = pd.read_parquet(METADATA_PROCESSED / f'metadata_clean_{DATA_TYPE[:3]}.parquet')
check(df_meta is not None, "Metadata loaded successfully")
check(len(df_meta) > 0, "Metadata dataframe is not empty")
logger.info(f"Metadata loaded, shape: {df_meta.shape}")

# Load signals
logger.info(f"Loading {DATA_TYPE} signals...")
df_signals = pd.read_parquet(SIGNALS_PROCESSED_IMPORT / f'{DATA_TYPE[:3]}_preprocessed_scaling.parquet')
check(df_signals is not None, "Signals loaded successfully")
check(len(df_signals) > 0, "Signals dataframe is not empty")
logger.info(f"Signals loaded, shape: {df_signals.shape}")
logger.info(f"Unique files: {df_signals['file'].nunique()}")

## 4. Metadata Preparation

Reduce metadata to **1 row per station** (removing component replication). This station-level DataFrame will be expanded back to component-level after PhaseNet detection.

**Note:** Unlike AR-AIC (which requires crustal velocities and search windows), PhaseNet operates directly on waveforms without prior velocity model information.

In [ ]:
# Exclude problematic stations
if EVENT_ID == 'IT-2009-0009':
    stations_to_exclude = ['TTS', 'GSG']  # TTS has very low SNR, GSG has inconsistent metadata
    df_meta = df_meta[~df_meta['STATION_CODE'].isin(stations_to_exclude)].copy()
    df_signals = df_signals[
        df_signals['file'].apply(get_station_from_filename).isin(
            df_meta['STATION_CODE'].unique()
        )
    ].copy()
    missing_time = df_meta[df_meta['DATE_TIME_FIRST_SAMPLE'].isna()]
    if len(missing_time) > 0:
        logger.warning(f"Excluding {len(missing_time)} stations with missing DATE_TIME_FIRST_SAMPLE: {missing_time['STATION_CODE'].tolist()}")
        df_meta = df_meta[df_meta['DATE_TIME_FIRST_SAMPLE'].notna()].copy()
    logger.info(f"Excluded stations: {stations_to_exclude}")
    logger.info(f"Remaining: {df_meta['STATION_CODE'].nunique()} stations, {df_signals['file'].nunique()} files")

In [ ]:
logger.info("Preparing station metadata (1 row per station)...")

# Select essential columns and reduce to 1 row per station
df_meta_stations = df_meta.drop_duplicates('STATION_CODE')[[
    'STATION_CODE',
    'STATION_LATITUDE_DEGREE',
    'STATION_LONGITUDE_DEGREE',
    'EPICENTRAL_DISTANCE_KM',
    'INSTRUMENTAL_FREQUENCY_HZ',
    'LOW_CUT_FREQUENCY_HZ',
    'HIGH_CUT_FREQUENCY_HZ',
    PEAK_COLUMN,
    TIME_PEAK_COLUMN,
    'EVENT_DATE',
    'EVENT_DEPTH_KM',
    'DATE_TIME_FIRST_SAMPLE'
]].copy()

logger.info("Calculating hypocentral distance for QC...")
hypo_depth_km = df_meta_stations['EVENT_DEPTH_KM'].iloc[0]
df_meta_stations = add_hypocentral_distance(
    df_meta_stations,
    hypo_depth_km=hypo_depth_km,
    distance_col='EPICENTRAL_DISTANCE_KM'
)

check('hypocentral_distance_km' in df_meta_stations.columns, "Hypocentral distance calculated")

n_stations = len(df_meta_stations)
logger.info(f"Station metadata ready: {n_stations} unique stations")

# Display first rows
print("\nFirst 5 stations:")
display(df_meta_stations.head())

# Summary statistics
print("\nEpicentral distance range:")
print(f"  Min: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].min():.2f} km")
print(f"  Max: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].max():.2f} km")
print(f"  Median: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].median():.2f} km")

## 5. Crustal Velocity Estimation

Although PhaseNet does **not require** crustal velocities for phase detection (it learns features directly from waveforms), we compute CRUST1.0 velocities here for **quality control** purposes:
- Compare PhaseNet picks vs. theoretical arrivals
- Assess velocity model accuracy
- Identify systematic biases

See notebook `03a_phase_identification_ar_pick` for detailed explanation of CRUST1.0 querying.

In [ ]:
# Extract hypocenter depth (assuming single event)
hypo_depth = df_meta_stations['EVENT_DEPTH_KM'].iloc[0]

# Add crustal velocities (vp_crust, vs_crust)
df_meta_stations = add_crustal_velocities(
    df_meta_stations,
    hypo_depth_km=hypo_depth,
    lat_col='STATION_LATITUDE_DEGREE',
    lon_col='STATION_LONGITUDE_DEGREE'
)
check('vp_crust' in df_meta_stations.columns, "vp_crust column added")
check('vs_crust' in df_meta_stations.columns, "vs_crust column added")

## 6. Theoretical Arrival Times

Compute theoretical P/S arrivals using CRUST1.0 velocities (same as notebook 03a). These are used **only for validation**, not for guiding PhaseNet detection.

**Key difference from AR-AIC:**
- AR-AIC: theoretical times define **search windows** → required for detection
- PhaseNet: theoretical times only for **comparison** → optional for validation

See notebook 03a for detailed formulas and methodology.

In [ ]:
logger.info("Calculating theoretical arrival times...")

# Add theoretical arrivals (t_p_theo, t_s_theo)
df_meta_stations = add_theoretical_arrivals(
    df_meta_stations,
    hypo_depth_km=hypo_depth,
    distance_col='EPICENTRAL_DISTANCE_KM'
)

check('t_p_theo_seconds' in df_meta_stations.columns, "t_p_theo_seconds column added")
check('t_s_theo_seconds' in df_meta_stations.columns, "t_s_theo_seconds column added")
check('t_p_theo_samples' in df_meta_stations.columns, "t_p_theo_samples column added")
check('t_s_theo_samples' in df_meta_stations.columns, "t_s_theo_samples column added")

# Summary
print("\nTheoretical arrival time ranges:")
print(f"  P-wave: {df_meta_stations['t_p_theo_seconds'].min():.2f} - {df_meta_stations['t_p_theo_seconds'].max():.2f} s")
print(f"  S-wave: {df_meta_stations['t_s_theo_seconds'].min():.2f} - {df_meta_stations['t_s_theo_seconds'].max():.2f} s")

## 7. Load PhaseNet Model — INSTANCE Pre-trained

Load the **PhaseNet model pre-trained on INSTANCE dataset** via SeisBench framework.

### PhaseNet Architecture

PhaseNet (Zhu & Beroza, 2019) is a deep convolutional neural network (CNN) for seismic phase detection:

**Input:** 3-component waveform (HNE, HNN, HNZ) at 100 Hz
**Output:** 3 probability time series:
- P-wave probability $P(t)$ — likelihood of P-wave onset at each time $t$
- S-wave probability $S(t)$ — likelihood of S-wave onset
- Noise probability $N(t)$ — likelihood of noise (no phase)

**Architecture:**
- **Encoder:** 5 convolutional layers → downsample waveform, extract features
- **Bottleneck:** Dense layer → compress feature space
- **Decoder:** 5 transpose-convolutional layers → upsample to original time resolution
- **Output:** Softmax layer → 3 probability channels (P, S, N)

**Key features:**
- U-Net structure (encoder-decoder with skip connections)
- Trained end-to-end (no hand-crafted features)
- Processes entire waveform in one forward pass (no sliding window)
- Outputs continuous probability functions (not just single onset times)

### INSTANCE Dataset

INSTANCE (Italian Seismic dataset for machine learning) contains:
- **~54,000 waveforms** from Italian Seismic Network
- **Magnitude range:** 3.0-6.5
- **Distance range:** 0-400 km
- **Manual phase picks** by expert analysts (ground truth)

**Why INSTANCE for this study?**
- Regional/local earthquakes (similar to our Mw 3.8 event)
- Italian geology (Alps, sedimentary basins) → matches our study area
- Strong-motion data included → suitable for accelerograms

### Pre-trained Model

```python
model = sbm.PhaseNet.from_pretrained("instance")
```

**Model details:**
- Trained on INSTANCE waveforms at **100 Hz**
- 3-component input required (vertical + 2 horizontal)
- No fine-tuning applied (transfer learning from INSTANCE → our data)

**Performance (INSTANCE test set):**
- P-wave detection: ~98% recall, ~0.2s mean absolute error
- S-wave detection: ~95% recall, ~0.3s mean absolute error

### Inference Process

1. **Resample** original signal (200 Hz → 100 Hz)
2. **Normalize** by trace maximum (standard PhaseNet preprocessing)
3. **Pad** if needed (PhaseNet requires minimum 3000 samples = 30s at 100 Hz)
4. **Forward pass** through network → probability functions $P(t)$, $S(t)$, $N(t)$
5. **Peak detection:** Find maximum probability above threshold
   - Default thresholds: $P(t) > 0.1$ for P, $S(t) > 0.07$ for S
6. **Convert** onset indices back to original time base (account for resampling + offset)

### Advantages vs. Traditional Methods

**PhaseNet strengths:**
- No velocity model required
- No search windows needed
- Handles complex waveforms (emergent arrivals, noise)
- Consistent performance across distance/magnitude
- Trained on large dataset (generalizes well)

**PhaseNet limitations:**
- Requires resampling to 100 Hz (may introduce artifacts)
- Black-box model (hard to interpret failures)
- No uncertainty quantification (single probability peak)
- Performance depends on training data similarity

### References

**Zhu, W., & Beroza, G. C.** (2019). *PhaseNet: a deep-neural-network-based seismic arrival-time picking method.* Geophysical Journal International, 216(1), 261-273. DOI: [10.1093/gji/ggy423](https://doi.org/10.1093/gji/ggy423)

**Woollam, J., et al.** (2022). *SeisBench—A Toolbox for Machine Learning in Seismology.* Seismological Research Letters, 93(3), 1695-1709. DOI: [10.1785/0220210324](https://doi.org/10.1785/0220210324)

**Michelini, A., et al.** (2021). *INSTANCE—the Italian Seismic dataset for machine learning.* Earth System Science Data, 13, 5509-5544. DOI: [10.5194/essd-13-5509-2021](https://doi.org/10.5194/essd-13-5509-2021)

In [ ]:
logger.info("Loading PhaseNet model pre-trained on INSTANCE...")
model = sbm.PhaseNet.from_pretrained("instance")
check(model is not None, "PhaseNet model loaded successfully")

## 8. Phase Picking with PhaseNet

Apply PhaseNet to all signals with automatic resampling to 100 Hz (model training rate).

### Resampling Strategy

**Original data:** 200 Hz  
**PhaseNet input:** 100 Hz (decimation factor = 2)

**Resampling method:** Fourier-domain resampling (anti-aliasing)
1. Apply low-pass filter at Nyquist frequency (50 Hz)
2. Decimate by factor of 2
3. Preserve phase information

**Why 100 Hz?**
- PhaseNet trained on 100 Hz data (INSTANCE standard)
- Running on 200 Hz would require re-training
- 100 Hz sufficient for regional phases (main energy < 20 Hz)

### Detection Thresholds

**Probability thresholds:**
- `min_p_probability = 0.1` — Accept P picks with $P(t_{\\max}) > 0.1$
- `min_s_probability = 0.07` — Accept S picks with $S(t_{\\max}) > 0.07$

**Threshold selection rationale:**
- Lower thresholds → higher recall (fewer missed phases) but more false positives
- Higher thresholds → higher precision but more missed phases
- Default values (0.1 for P, 0.07 for S) balance recall/precision

**S-wave threshold lower than P:**
S-waves are typically more emergent and harder to detect → lower threshold compensates.

### Quality Filtering

After detection, picks are filtered to remove:
1. **Missing onsets:** Stations where $P(t)$ or $S(t)$ never exceeded threshold (NaN)
2. **Invalid ordering:** Stations where $t_S \leq t_P$ (physically impossible)
3. **Low confidence:** (Optional) stations with very low max probabilities

**Expected failure rate:** ~5-15% of stations (distant, low SNR, complex waveforms)

### Output Format

PhaseNet picks are stored with:
- **Onset times:** `t_p_seconds`, `t_s_seconds`, `t_p_samples`, `t_s_samples`
- **Probabilities:** `p_probability_max`, `s_probability_max` (confidence scores)
- **Station info:** `station_code`, `component`

**Dual representation:** Both sample indices (for slicing) and seconds (for visualization).

In [ ]:
# Phase picking with PhaseNet
logger.info(f"Starting PhaseNet inference on {DATA_TYPE} signals...")

df_picks = apply_phasenet_to_signals(
    df_signals=df_signals,
    model=model,
    signal_column=SIGNAL_COLUMN,
    sampling_rate_original=200,
    sampling_rate_target=100,
    min_p_probability=0,
    min_s_probability=0
)

logger.info(f"PhaseNet inference complete: {len(df_picks)} stations processed")

# Quality check and filtering
logger.info("Checking PhaseNet picks quality...")
n_initial = len(df_picks)

# Filter 1: missing onsets
df_picks = df_picks[
    df_picks['t_p_seconds'].notna() &
    df_picks['t_s_seconds'].notna()
].copy()
n_after_nan = len(df_picks)
if n_initial > n_after_nan:
    logger.warning(f"Filtered out {n_initial - n_after_nan} station(s) with missing onsets")

# Filter 2: S ≤ P
invalid_sp = df_picks[df_picks['t_s_seconds'] <= df_picks['t_p_seconds']]
if len(invalid_sp) > 0:
    logger.warning(f"{len(invalid_sp)} stations with S ≤ P — excluded:")
    logger.warning(str(invalid_sp[['station_code', 't_p_seconds', 't_s_seconds']]))
    df_picks = df_picks[df_picks['t_s_seconds'] > df_picks['t_p_seconds']].copy()


# Extract valid station list
valid_stations = df_picks['station_code'].unique().tolist()

# Summary
valid_stations = df_picks['station_code'].unique().tolist()

logger.info(f"Filtering summary: {n_initial} → {len(df_picks)} stations retained")
logger.info(f"\nPhaseNet picks summary (after all filters):")
logger.info(f"  Valid stations: {len(valid_stations)}")
logger.info(f"  Mean P probability: {df_picks['p_probability_max'].mean():.3f}")
logger.info(f"  Mean S probability: {df_picks['s_probability_max'].mean():.3f}")
logger.info(f"  Mean S-P time: {(df_picks['t_s_seconds'] - df_picks['t_p_seconds']).mean():.2f}s")

output_path = SIGNALS_PROCESSED_EXPORT / f'phasenet_picks_{DATA_TYPE}.csv'
df_picks.to_csv(output_path, index=False)
logger.info(f"\nSaved: {output_path}")

## 9. Merge PhaseNet Picks with Metadata

Combine PhaseNet phase picks with station metadata to create a complete DataFrame for analysis.

### Merging Process

The merge operation:
1. **Joins** PhaseNet picks (station-level) with metadata (station-level)
2. **Calculates** origin time from first sample timestamp and event time
3. **Adds** epicentral distance, PGA/PGV/PGD, filter parameters
4. **Preserves** all PhaseNet probability scores

**Key difference from AR-AIC:**
PhaseNet outputs are **already station-level** (no multi-component combination needed), so the merge is simpler.

### Origin Time Calculation

Origin time ($t_0$) is the time from first sample to earthquake origin:

$$t_0 = (T_{\text{event}} - T_{\text{first sample}}) \times 1\text{ second}$$

where:
- $T_{\text{event}}$ = earthquake origin time (from catalog)
- $T_{\text{first sample}}$ = timestamp of first sample in recording

**Purpose:** Convert absolute timestamps to relative times (needed for theoretical arrival calculations).

### Residual Calculation

Compute residuals between PhaseNet picks and CRUST1.0 theoretical arrivals:

$$
\begin{aligned}
\Delta t_P &= t_{P,\text{PhaseNet}} - t_{P,\text{theo}} \\\\
\Delta t_S &= t_{S,\text{PhaseNet}} - t_{S,\text{theo}}
\end{aligned}
$$

**Expected residuals:**
- **Systematic bias:** PhaseNet typically picks **3-5 seconds early** (trained on manually picked catalogs, which tend to pick at signal onset rather than first break)
- **Standard deviation:** ±0.5-1.5 seconds (picking uncertainty + velocity model error)

**Interpretation:**
- Negative residuals → PhaseNet picks earlier than theoretical
- Positive residuals → PhaseNet picks later than theoretical
- Large scatter → velocity model inaccurate or complex wave propagation

### Output Columns

The merged DataFrame contains:
- **Metadata:** Station coordinates, distances, PGA/PGV/PGD
- **PhaseNet picks:** `t_p_seconds`, `t_s_seconds` (both samples and seconds)
- **Theoretical arrivals:** `t_p_theo_seconds`, `t_s_theo_seconds`
- **Residuals:** `p_residual`, `s_residual`
- **Confidence scores:** `p_probability_max`, `s_probability_max`
- **Crustal velocities:** `vp_crust`, `vs_crust` (for QC)

In [ ]:
logger.info("Merging PhaseNet picks with station metadata...")

df_meta_stations = merge_phasenet_picks_with_metadata(df_picks, df_meta_stations, sampling_rate=200)

check(len(df_meta_stations) > 0, f"Metadata merged: {len(df_meta_stations)} stations")
check('origin_time_seconds' in df_meta_stations.columns, "origin_time calculated from metadata")

logger.info(f"Origin time range: {df_meta_stations['origin_time_seconds'].min():.2f}s - {df_meta_stations['origin_time_seconds'].max():.2f}s")

In [ ]:
# Calculate residuals
logger.info("Calculating onset residuals...")

# P residuals
df_meta_stations['p_residual_seconds'] = (
    df_meta_stations['t_p_detected_seconds'] - 
    df_meta_stations['t_p_theo_seconds']
)

# S residuals
df_meta_stations['s_residual_seconds'] = (
    df_meta_stations['t_s_detected_seconds'] - 
    df_meta_stations['t_s_theo_seconds']
)

df_meta_stations.loc[
    df_meta_stations['t_p_theo_seconds'] < 0, 'p_residual_seconds'
] = np.nan
df_meta_stations.loc[
    df_meta_stations['t_s_theo_seconds'] < 0, 's_residual_seconds'
] = np.nan

# Summary
logger.info(f"\nResidual statistics:")
logger.info(f"  P residual: {df_meta_stations['p_residual_seconds'].mean():+.2f} ± {df_meta_stations['p_residual_seconds'].std():.2f} s")
logger.info(f"  S residual: {df_meta_stations['s_residual_seconds'].mean():+.2f} ± {df_meta_stations['s_residual_seconds'].std():.2f} s")
logger.info(f"  P range: [{df_meta_stations['p_residual_seconds'].min():+.2f}, {df_meta_stations['p_residual_seconds'].max():+.2f}] s")
logger.info(f"  S range: [{df_meta_stations['s_residual_seconds'].min():+.2f}, {df_meta_stations['s_residual_seconds'].max():+.2f}] s")

check('p_residual_seconds' in df_meta_stations.columns, "p_residual_seconds column added")
check('s_residual_seconds' in df_meta_stations.columns, "s_residual_seconds column added")

## 10. Convert Signals to Dictionary

Convert long-format DataFrame to nested dictionary structure for efficient windowing operations.

**Structure:**
```python
signals_dict = {
    'STATION_HNE.ASC': array([...]),  # 1D numpy array per file
    'STATION_HNN.ASC': array([...]),
    'STATION_HNZ.ASC': array([...]),
    ...
}
```

This format is required by `segment_all_signals()` and `add_coda_onsets_to_dataframe()`.

In [ ]:
logger.info("Adding time column to signals...")
df_signals = add_time_columns(
    df_signals, 
    df_meta, 
    time_col='DATE_TIME_FIRST_SAMPLE',
    sampling_interval_col='SAMPLING_INTERVAL_S'
)
check('time' in df_signals.columns, "Time column added to signals")

# Filter signals to only include valid stations
logger.info(f"Filtering signals to valid stations ({len(valid_stations)} stations)...")
n_total_rows = len(df_signals)
df_signals = df_signals[
    df_signals['file'].apply(get_station_from_filename).isin(valid_stations)
].copy()
logger.info(f"Filtered signals: {len(df_signals)} rows (from {n_total_rows} total)")

# Convert DataFrame to dict
signals_dict = convert_signals_to_dict(df_signals,
                                       signal_column=SIGNAL_COLUMN)

check(len(signals_dict) > 0, "Signals dictionary created")
logger.info(f"Dictionary contains {len(signals_dict)} stations")

# Validate structure
print("\nValidating signals dictionary...")
report = validate_signals_dict(signals_dict)

check(report['valid'], "All signals validated successfully")

# Save signals_dict for sensitivity analysis
output_file = SIGNALS_PROCESSED_EXPORT / f'signals_{DATA_TYPE}_dict.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(signals_dict, f)
logger.info(f"Saved signals_dict: {output_file}")
logger.info(f"File size: {output_file.stat().st_size / 1e6:.2f} MB")

## 11. Expand DataFrame to Component Level

Expand station-level DataFrame (1 row per station) to **component-level** (3 rows per station: HNE, HNN, HNZ) to accommodate coda onset times.

**Why expand?**
- PhaseNet picks are station-level (network outputs single P/S time per station)
- Coda detections are component-level (envelope/energy may differ across components)
- Moment scaling treats components as independent ensemble members

**Process:**
1. Replicate each station row 3 times
2. Add `STREAM_COMPONENT` column (HNE, HNN, HNZ)
3. Update `file` column to match original filenames
4. Preserve all PhaseNet pick columns (same for all 3 components per station)

**Result:** `df_full` with `N_stations × 3` rows, ready for coda onset addition.

**Note:** PhaseNet processes components together internally but outputs single onset per station, unlike AR-AIC which explicitly uses multi-component characteristic functions.

In [ ]:
logger.info("Expanding metadata to component level...")

# Filter df_meta to only include valid stations
df_meta_valid = df_meta[df_meta['STATION_CODE'].isin(valid_stations)].copy()
logger.info(f"Filtered df_meta: {len(df_meta_valid)} components ({len(df_meta)} total)")

# Expand using filtered metadata
df_full = expand_to_component_level(df_meta_stations, df_meta_valid)

# Dynamic check based on valid_stations
expected_components = len(valid_stations) * 3
check(
    len(df_full) == expected_components, 
    f"Expanded to component level: {len(df_full)} components "
    f"(expected {expected_components} from {len(valid_stations)} stations)"
)

logger.info(f"Component-level DataFrame shape: {df_full.shape}")

## 12. Coda Onset Detection

Detect coda onsets using the **same four methods** as notebook 03a:

1. **Rautian (1978)** — Theoretical: $t_{\text{coda}} = 2 \times t_S - t_0$
2. **Arias Intensity (D5-95)** — Energy-based: 95% cumulative energy threshold
3. **Envelope Decay** — Amplitude-based: envelope drops below 25% of peak
4. **Median** — Robust combination: median of the three methods above

**Why same methods?**
To enable **direct comparison** of moment scaling results between AR-AIC and PhaseNet. Using identical coda detection ensures differences arise only from P/S picking, not from window definition.

### Expected Coda Onset Differences

Because PhaseNet picks P and S **earlier** than AR-AIC (typically 3-5 seconds), coda onsets will also be earlier:

- **Rautian method:** Directly depends on $t_S$ → systematic shift of ~3-5s earlier
- **Arias method:** Independent of picks (uses full waveform) → no systematic shift
- **Envelope method:** Depends on S-window definition → similar shift to Rautian
- **Median:** Intermediate behavior

**Implication for moment scaling:**
PhaseNet-based windows may include **more S-wave energy in coda window** due to early picks. This could affect coda scaling exponents ζ(q).

### Implementation

See notebook **03a Section 8.2** for detailed explanation of each coda detection method, including:
- Physical basis
- Algorithm details
- Advantages and limitations
- Expected performance

The implementation here is **identical** to 03a — only the input onset times differ.

In [ ]:
logger.info("Calculating coda onsets for all methods...")

# Check for missing onsets
n_total = len(df_full)
n_missing_p = df_full['t_p_detected_seconds'].isna().sum()
n_missing_s = df_full['t_s_detected_seconds'].isna().sum()

if n_missing_p > 0 or n_missing_s > 0:
    logger.warning(f"Missing onsets: P={n_missing_p}, S={n_missing_s}")
    logger.warning("Filtering out components with missing onsets...")
    
    df_full_valid = df_full[
        df_full['t_p_detected_seconds'].notna() & 
        df_full['t_s_detected_seconds'].notna()
    ].copy()
    
    logger.info(f"Valid components: {len(df_full_valid)}/{n_total}")
else:
    df_full_valid = df_full.copy()

# Coda detection
df_full = add_coda_onsets_to_dataframe(
    df_full_valid, 
    signals_dict,
    threshold_arias=0.95,
    threshold_envelope=0.3,
    sampling_rate=200,
    unit='samples'
)

check('t_coda_rautian_seconds' in df_full.columns, "Coda onsets computed")
logger.info("Coda onsets computed for all methods: rautian, arias, envelope, median")

# Summary
print("\nCoda onset summary (mean S-wave duration):")
for method in ['rautian', 'arias', 'envelope', 'median']:
    col = f's_duration_{method}_seconds'
    if col in df_full.columns:
        mean_dur = df_full[col].mean()
        std_dur = df_full[col].std()
        print(f"  {method.capitalize():10s}: {mean_dur:.2f} ± {std_dur:.2f}s")

In [ ]:
onset_figs= plot_onset_detection_results(
    signals_dict,
    df_meta_stations,
    method='phasenet',
    stations=None,
    output_dir=FIGURES_DIR / 'onset_detection'
)

In [ ]:
# Save df_full for sensitivity analysis
df_full_output_file = SIGNALS_PROCESSED_EXPORT / f'df_full_{DATA_TYPE}_phasenet.parquet'
df_full.to_parquet(df_full_output_file, index=False)
logger.info(f"Saved df_full: {df_full_output_file}")
logger.info(f"Shape: {df_full.shape}")
logger.info(f"Columns: {len(df_full.columns)}")

## Post event onset detection

In [ ]:
df_full = add_coda_end_to_dataframe(
    df_full,
    signals_dict,
    coda_methods=['rautian', 'arias', 'envelope', 'median'],
    threshold_factor=0.10,
    stability_duration=2.0,
    sampling_rate=200,
    smoothing_window=0.5
)

## 13. Window Segmentation

Segment signals into **four non-overlapping windows** using PhaseNet picks:

$$
\begin{aligned}
\text{Pre-event:} & \quad [t_{\text{start}}, t_P) \\\\
\text{P-wave:} & \quad [t_P, t_S) \\\\
\text{S-wave:} & \quad [t_S, t_{\text{coda}}) \\\\
\text{Coda:} & \quad [t_{\text{coda}}, t_{\text{end}}]
\end{aligned}
$$

**Key difference from AR-AIC:**
- PhaseNet picks are **3-5 seconds earlier** → windows shift earlier in time
- **Pre-event window longer** (more noise samples)
- **P-wave window slightly longer** (earlier P pick)
- **S-wave window slightly shorter** (earlier S pick, earlier coda onset)

### Output Structure

Identical to notebook 03a: nested dictionaries with station → component → window → data.

Four separate windowed files created (one per coda method):
- `windowed_{signal_type}_rautian_phasenet.pkl`
- `windowed_{signal_type}_arias_phasenet.pkl`
- `windowed_{signal_type}_envelope_phasenet.pkl`
- `windowed_{signal_type}_median_phasenet.pkl`

**Naming convention:** Files end with `_phasenet.pkl` (vs. `_ar_pick.pkl` for AR-AIC) to distinguish picker methods.

See notebook **03a Section 10** for detailed window definitions and data structure.

In [ ]:
for method in ['rautian', 'arias', 'envelope', 'median']:
    logger.info(f"Segmenting signals with {method} coda method...")
    
    # Windowing
    windowed_signals = segment_all_signals(
        signals_dict, 
        df_full, 
        coda_method=method,
        pre_p_duration = 'full',
        min_window_duration = 2.0
    )
    
    # Save
    output_file = SIGNALS_PROCESSED_EXPORT / f'windowed_{DATA_TYPE}_{method}_phasenet.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(windowed_signals, f)
    
    logger.info(f"Saved: {output_file}")
    logger.info(f"File size: {output_file.stat().st_size / 1e6:.2f} MB")


    all_stations = list(windowed_signals.keys())
    plot_multiple_stations(
    stations=all_stations,
    signals_dict=signals_dict,
    windowed_signals=windowed_signals,
    df_onsets=df_full,
    coda_method=method,
    output_dir= FIGURES_DIR / f'windows_{method}',
    close_after_save=True 
)

## 14. Quality Control

Apply the **same three validation checks** as notebook 03a to enable direct comparison:

### Check 1: Peak Timing
**Criterion:** Peak ground motion should occur in S-wave window.

**Expected performance difference:**
- AR-AIC: ~90% pass rate (picks near peak by design)
- PhaseNet: ~85% pass rate (earlier picks → peak may fall just outside S-window)

---

### Check 2: Monotonicity with Distance
**Criterion:** Arrival times should increase with hypocentral distance.

**Test:** Sort stations by distance, check for violations where $d_i < d_j$ but $t_i > t_j$.

**Expected performance difference:**
- AR-AIC P: ~90% pass rate (search windows constrain picks)
- PhaseNet P: ~85% pass rate (no distance constraint, pure waveform-based)
- AR-AIC S: ~85% pass rate
- PhaseNet S: ~80% pass rate

**Why PhaseNet may violate more:**
PhaseNet has no velocity model constraint → picks purely on waveform features. If lateral velocity heterogeneity exists, PhaseNet may reflect actual travel time variations that violate the 1D model assumption.

---

### Check 3: Signal-to-Noise Ratio (SNR)
**Criterion:** Phase window SNR ≥ 3.

$$\text{SNR} = \frac{\text{RMS}_{\text{signal}}}{\text{RMS}_{\text{noise}}}$$

See notebook **03a Section 11** for detailed explanation of each quality check.

In [ ]:
logger.info("Running quality control on windowed signals...")
logger.info("=" * 80)

# Run QC per ogni metodo di coda
qc_results = {}

for method in ['rautian', 'arias', 'envelope', 'median']:
    logger.info(f"\n{method.upper()} METHOD")
    logger.info("-" * 40)
    
    # Load windowed signals
    input_file = SIGNALS_PROCESSED_EXPORT / f'windowed_{DATA_TYPE}_{method}_phasenet.pkl'
    
    if not input_file.exists():
        logger.warning(f"File not found: {input_file}")
        continue
    
    with open(input_file, 'rb') as f:
        windowed_signals = pickle.load(f)
    
    # Run QC
    qc_result = quality_control_all_stations(
        windowed_signals,
        df_full,
        df_meta_stations,
        peak_column=PEAK_COLUMN,
        time_peak_column=TIME_PEAK_COLUMN,
        snr_threshold=3.0,
        coda_method=method
    )
    
    qc_results[method] = qc_result
    
    # Print summary
    print_quality_control_summary(qc_result)

logger.info("\n" + "=" * 80)
logger.info("Quality control complete for all methods")

## 15. Summary

This notebook implemented PhaseNet-based phase detection as an alternative to AR-AIC (notebook 03a).

### Processing Pipeline

**Inputs:**
- Preprocessed signals (baseline-corrected, unnormalized)
- Station metadata with coordinates and epicentral distances

**Processing steps:**
1. Load PhaseNet model (INSTANCE pre-trained)
2. Resample signals (200 Hz → 100 Hz)
3. Phase detection (CNN-based picking)
4. Merge with metadata
5. Coda onset detection (4 methods)
6. Window segmentation (4 windows)
7. Quality control (3 validation checks)

**Outputs:**
- `df_full_{signal_type}_phasenet.parquet` — Complete metadata with PhaseNet picks
- `phasenet_picks_{signal_type}.csv` — Raw PhaseNet output
- `windowed_{signal_type}_{coda}_phasenet.pkl` — Segmented signals (4 files)
- Validation figures in `figures/03b_phase_identification_phasenet/{signal_type}/`

---

### Recommendations for Moment Scaling Analysis

**Use both methods:**
1. Run moment scaling on **both AR-AIC and PhaseNet windows**
2. Compare scaling exponents ζ(q) between methods
3. Assess sensitivity to window definition

**Expected impact:**
- **P-wave window:** Minimal difference (both methods similar)
- **S-wave window:** Moderate difference (PhaseNet shorter, may affect ζ(q))
- **Coda window:** Largest difference (timing shift affects Rautian/Envelope)

**If results disagree:**
- Check which method has better QC pass rate for this dataset
- Visual inspection of waveforms (which picker looks more accurate?)
- Consider ensemble: combine picks from both methods

---

### Next Steps

1. Compare PhaseNet vs. AR-AIC picks visually (scatter plots, residuals)
2. Run moment scaling analysis on both picker outputs (notebook 04a)
3. Assess impact of picker choice on scaling exponents
4. Decide on final picker for results (or present both)

**Hypothesis:** Moment scaling exponents should be **robust to moderate window shifts** (3-5 seconds), but we verify this by comparing both methods.